## Duplicate Rows

Duplicates are one of the most common data quality issues. They inflate counts, distort averages, and cause joins to fan out unexpectedly. There are two types to watch for:

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

### Exact duplicates
Every column in the row is identical. These are usually caused by a data loading error (e.g. a file was processed twice) or an overlapping snapshot.

### Duplicates on natural keys
The identifying columns match, but other columns differ. For example, two rows for the same `pupil_id` but with different `first_name` values. This often indicates:
* A data entry error (the same entity recorded inconsistently)
* A legitimate change over time that wasn’t modelled correctly (e.g. a name change without versioning)
* The grain is finer than you assumed (perhaps a date column should be part of the key)

### Strategy for handling duplicates

1. **Detect** — always check before assuming data is clean
2. **Understand** — are they genuine errors or a sign that the grain is wrong?
3. **Decide** — remove exact duplicates with `DISTINCT` or `ROW_NUMBER()`; for key-level duplicates, define a business rule (e.g. keep the latest record)

We’ll use `pupils_autumn_2024` from `catalog_40_copper_analyst_training.messy_data`, which contains both types of duplicate.

In [0]:
-- Preview the pupils table — can you spot the duplicates?
SELECT *
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024
ORDER BY pupil_id;

### Detect exact duplicates

In [0]:
-- Detect EXACT duplicates: rows where every column is identical
-- GROUP BY all columns and look for counts > 1
SELECT
  pupil_id
  ,first_name
  ,last_name
  ,gender
  ,date_of_birth
  ,school_urn
  ,year_group
  ,COUNT(*) as occurrence_count
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024
GROUP BY pupil_id, first_name, last_name, gender, date_of_birth, school_urn, year_group
HAVING COUNT(*) > 1;

### Detect duplicates on natural keys

In [0]:
-- Detect duplicates on NATURAL KEY (pupil_id)
-- These rows share the same pupil_id but differ in other columns
SELECT
  pupil_id
  ,COUNT(*) as row_count
  ,COUNT(DISTINCT first_name) as distinct_first_names
  ,COUNT(DISTINCT last_name) as distinct_last_names
  ,COUNT(DISTINCT school_urn) as distinct_schools
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024
GROUP BY pupil_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

In [0]:
-- Remove duplicates using ROW_NUMBER: keep one row per pupil_id
-- Business rule: prefer the row with the most complete data (non-null date_of_birth)
-- then alphabetical first_name as a tie-breaker
SELECT *
FROM (
  SELECT
    *
    ,ROW_NUMBER() OVER (
      PARTITION BY pupil_id
      ORDER BY
        CASE WHEN date_of_birth IS NOT NULL THEN 0 ELSE 1 END
        ,first_name ASC
    ) as rn
  FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024
)
WHERE rn = 1
ORDER BY pupil_id;

### Going further: detecting duplicates dynamically

The basic approach above requires you to list every column manually in the `GROUP BY`. For wide tables with dozens of columns, this is tedious and error-prone.

Using `INFORMATION_SCHEMA.COLUMNS`, you can **dynamically generate** the exact-duplicate detection query for any table — regardless of how many columns it has. This is especially useful when building reusable data quality checks.

In [0]:
-- Dynamically build an exact-duplicate detection query for any table
-- This reads the column list from INFORMATION_SCHEMA and constructs the GROUP BY / HAVING

DECLARE dup_sql STRING;
DECLARE target_table STRING DEFAULT 'pupils_autumn_2024';

SET VAR dup_sql = (
  WITH cols AS (
    SELECT column_name
    FROM catalog_40_copper_analyst_training.information_schema.columns
    WHERE table_schema = 'messy_data'
      AND table_name = target_table
      AND column_name != 'metadata_json'  -- exclude complex types from grouping
    ORDER BY ordinal_position
  )
  SELECT concat(
    'SELECT '
    ,aggregate(collect_list(concat('`', column_name, '`')), '', (acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END)
    ,', COUNT(*) as occurrence_count'
    ,' FROM catalog_40_copper_analyst_training.messy_data.', target_table
    ,' GROUP BY '
    ,aggregate(collect_list(concat('`', column_name, '`')), '', (acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END)
    ,' HAVING COUNT(*) > 1'
    ,' ORDER BY occurrence_count DESC'
  )
  FROM cols
);

EXECUTE IMMEDIATE dup_sql;

> **Tip:** You can change the `target_table` variable to any table in the `messy_data` schema to run the same duplicate check. Try setting it to `'assessments'` or `'schools_autumn_2024'` to explore other tables.